In [ ]:
# %%capture
!pip uninstall -y paddlepaddle paddleocr paddlex
!pip install paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
# !pip install paddlepaddle==3.2.2
!pip install paddleocr
!pip install --quiet vietocr
!pip install qwen-vl-utils
!pip install -U ultralytics

# ---------------------------READ ME FIRST!!!!--------------------------------------
# run this cell then restart the kernel
# In kaggle you could do: click "Run" -> choose "Restart & clear cell outputs"
# Then just run other cells, dont run this cell again
# ----------------------------------------------------------------------------------

# Import libraries

In [ ]:
from pathlib import Path
import random
import cv2
# from paddleocr import PaddleOCR
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import AutoProcessor, Blip2ForConditionalGeneration
import torch
import json

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/khngxuninh/autoshot-output/shot_segments.csv")
df.shape

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path


@dataclass
class FeatureConfig:
    frame_root: Path = Path("/kaggle/input/datasets/khngxuninh/autoshot-output/frames")
    # output_path: Path = Path("/kaggle/working/annotations.jsonl")
    output_root: Path = Path("/kaggle/working")
    # video_ids: list[str] = field(default_factory=lambda: ["L30_V001", "L30_V002"])
    video_ids: list[str] = field(default_factory=list)

    use_caption: bool = True
    use_ocr: bool = True
    use_objects: bool = True

    captioning_model: str = "blip"  # "qwen" or "blip"
    ocr_model: str = "vietocr"      # "vietocr" or "qwen"
    vietocr_model: str = "vgg_seq2seq"  # "vgg_seq2seq" or "vgg_transformer"
    yolo_model: str = "yolo12s.pt"  # use "yolo12n.pt" for faster, lower-quality objects
    yolo_imgsz: int = 640

    combine_qwen_caption_ocr: bool = False
    qwen_max_pixels: int = 1024 * 1024
    caption_max_new_tokens: int = 80
    blip_min_new_tokens: int = 10
    blip_num_beams: int = 1
    blip_repetition_penalty: float = 1.2
    blip_length_penalty: float = 1.1
    ocr_max_new_tokens: int = 128
    combined_max_new_tokens: int = 192

    caption_batch_size: int = 8
    qwen_ocr_batch_size: int = 16
    combined_qwen_batch_size: int = 8
    ocr_det_batch_size: int = 16
    ocr_recog_batch_size: int = 64
    yolo_batch_size: int = 32
    pipeline_batch_size: int = 64

    overwrite_output: bool = True
    skip_existing: bool = True
    save_every_n_batches: int = 5
    test_image_path: str = "/kaggle/input/datasets/khngxuninh/autoshot-output/frames/L30_V001/shot_0000_last_f000074.jpg"
    test_image_path2: str = "/kaggle/input/datasets/khngxuninh/autoshot-output/frames/L30_V019/shot_0007_middle_f000583.jpg"


CFG = FeatureConfig()

test_image_path = CFG.test_image_path
test_image_path2 = CFG.test_image_path2

OCR_PROMPT = """
Hãy trích xuất toàn bộ chữ và chữ số xuất hiện trong ảnh.

Yêu cầu:
- Chỉ trả về một JSON array.
- Mỗi phần tử là một chuỗi văn bản xuất hiện trong ảnh.
- Giữ nguyên chữ hoa/thường, dấu tiếng Việt và dấu câu.
- Không dịch.
- Không giải thích.
- Không thêm markdown.
- Nếu không có chữ thì trả về [].

Ví dụ:
["Tuổi Trẻ TV", "tv.tuoitre.vn"]
""".strip()

CAPTIONING_MODEL = CFG.captioning_model
OCR_MODEL = CFG.ocr_model
VIETOCR_MODEL = CFG.vietocr_model

In [ ]:
CFG.video_ids = sorted([
    p.name
    for p in CFG.frame_root.iterdir()
    if p.is_dir()
])

print("Number of videos:", len(CFG.video_ids))
print(CFG.video_ids[:10])

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

In [ ]:
if CAPTIONING_MODEL == "qwen":
    CAPTIONING_PROMPT = """
    Bạn là hệ thống tạo caption cho ảnh/keyframe video.

    Nhiệm vụ:
    - Mô tả nội dung chính của ảnh bằng tiếng Việt.
    - Tập trung vào người, hành động, vật thể, bối cảnh, địa điểm, sự kiện.
    - Nếu có chữ/logo/biển báo/phụ đề trên ảnh, hãy ghi lại các chữ quan trọng nhìn thấy được.
    - Không suy đoán thông tin không chắc chắn.
    - Không mô tả quá dài.

    Định dạng trả về:
    Một đoạn văn tiếng Việt ngắn gọn, 1-3 câu.
    """.strip()
elif CAPTIONING_MODEL == "blip":
    CAPTIONING_PROMPT = (
        "a photo of"
    )

QWEN_COMBINED_PROMPT = """
Bạn là hệ thống phân tích ảnh/keyframe video.

Hãy trả về đúng một JSON object, không markdown, không giải thích:
{
  "caption": "Một đoạn mô tả ảnh bằng tiếng Việt, 1-3 câu.",
  "texts": ["các chữ xuất hiện trong ảnh, giữ nguyên dấu và chữ hoa/thường"]
}

Yêu cầu:
- Caption tập trung vào người, hành động, vật thể, bối cảnh, địa điểm, sự kiện.
- Không suy đoán thông tin không chắc chắn.
- Nếu không thấy chữ trong ảnh thì "texts" là [].
""".strip()

# Define models

### OCR

In [ ]:
def crop_text_line(image, box):
    box = np.array(box).astype(np.float32)

    w = int(max(
        np.linalg.norm(box[0] - box[1]),
        np.linalg.norm(box[2] - box[3])
    ))
    h = int(max(
        np.linalg.norm(box[0] - box[3]),
        np.linalg.norm(box[1] - box[2])
    ))

    dst = np.array([
        [0, 0],
        [w, 0],
        [w, h],
        [0, h]
    ], dtype=np.float32)

    M = cv2.getPerspectiveTransform(box, dst)
    crop = cv2.warpPerspective(image, M, (w, h))

    return crop

In [ ]:
if CFG.use_ocr and OCR_MODEL == "vietocr":
    config = Cfg.load_config_from_name(VIETOCR_MODEL)
    config["cnn"]["pretrained"] = False
    config["device"] = device

    predictor = Predictor(config)

    print("VietOCR loaded OK")
    # ocr = PaddleOCR(
    #     lang="vi",
    #     use_doc_orientation_classify=False,
    #     use_doc_unwarping=False,
    #     use_textline_orientation=False,
    # )
    # print("Paddle OCR loaded OK")
    from paddleocr import TextDetection

    detector = TextDetection(
        model_name="PP-OCRv5_mobile_det",
        device="gpu" if device == "cuda" else "cpu",
        limit_side_len=960,
        limit_type="max",
    )
    print("detector loaded okay")

### Image captioning

In [ ]:
from qwen_vl_utils import process_vision_info

In [ ]:
NEED_QWEN = (
    (CFG.use_caption and CAPTIONING_MODEL == "qwen")
    or (CFG.use_ocr and OCR_MODEL == "qwen")
)

if NEED_QWEN:
    caption_model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

    qwen = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        caption_model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    ).eval()

    qwen_processor = AutoProcessor.from_pretrained(caption_model_name)
    qwen_processor.tokenizer.padding_side = "left"

if CFG.use_caption and CAPTIONING_MODEL == "blip":
    processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
    blip = Blip2ForConditionalGeneration.from_pretrained(
        "Salesforce/blip2-opt-2.7b",
        torch_dtype=torch.float16,
    )
    blip.to(device).eval()

## Object Detection

In [ ]:
from ultralytics import YOLO

yolo = YOLO(CFG.yolo_model) if CFG.use_objects else None
if yolo is not None:
    try:
        yolo.fuse()
    except Exception:
        pass

# Helper functions

### OCR

In [ ]:
import numpy as np

def poly_to_xyxy(poly):
    poly = np.asarray(poly)
    x1 = np.min(poly[:, 0])
    y1 = np.min(poly[:, 1])
    x2 = np.max(poly[:, 0])
    y2 = np.max(poly[:, 1])
    return [x1, y1, x2, y2]

def merge_boxes_by_line(polys, y_thresh=25, x_gap_thresh=80):
    boxes = [poly_to_xyxy(p) for p in polys]
    boxes = sorted(boxes, key=lambda b: (b[1], b[0]))

    lines = []

    for box in boxes:
        x1, y1, x2, y2 = box
        cy = (y1 + y2) / 2

        matched = False

        for line in lines:
            lx1, ly1, lx2, ly2 = line["box"]
            lcy = (ly1 + ly2) / 2

            same_line = abs(cy - lcy) < y_thresh
            close_x = x1 - lx2 < x_gap_thresh

            if same_line and close_x:
                line["box"] = [
                    min(lx1, x1),
                    min(ly1, y1),
                    max(lx2, x2),
                    max(ly2, y2)
                ]
                matched = True
                break

        if not matched:
            lines.append({"box": box})

    return [line["box"] for line in lines]

def crop_xyxy(img_rgb, box, pad=8):
    h, w = img_rgb.shape[:2]
    x1, y1, x2, y2 = map(int, box)

    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(w, x2 + pad)
    y2 = min(h, y2 + pad)

    return img_rgb[y1:y2, x1:x2]

In [ ]:
def chunked(items, batch_size):
    items = list(items)
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def _build_qwen_messages(image_path, prompt):
    image_content = {"type": "image", "image": str(image_path)}
    if CFG.qwen_max_pixels:
        image_content["max_pixels"] = CFG.qwen_max_pixels

    return [
        {
            "role": "user",
            "content": [
                image_content,
                {"type": "text", "text": prompt},
            ],
        }
    ]


def _qwen_generate_batch(image_paths, prompt, max_new_tokens=128):
    conversations = [_build_qwen_messages(path, prompt) for path in image_paths]
    texts = [
        qwen_processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        for messages in conversations
    ]

    image_inputs = []
    video_inputs = []
    for messages in conversations:
        sample_images, sample_videos = process_vision_info(messages)
        if sample_images:
            image_inputs.extend(sample_images)
        if sample_videos:
            video_inputs.extend(sample_videos)

    inputs = qwen_processor(
        text=texts,
        images=image_inputs or None,
        videos=video_inputs or None,
        return_tensors="pt",
        padding=True,
    ).to(qwen.device)

    with torch.inference_mode():
        output_ids = qwen.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            use_cache=True,
        )

    generated_ids = [
        out[len(inp):]
        for inp, out in zip(inputs.input_ids, output_ids)
    ]
    return qwen_processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )


def _predict_text_detector(paths):
    paths = [str(path) for path in paths]
    try:
        return list(detector.predict(paths))
    except Exception:
        outputs = []
        for path in paths:
            outputs.extend(list(detector.predict(path)))
        return outputs


def _extract_polys(det_result):
    if isinstance(det_result, dict):
        boxes = det_result.get("dt_polys", [])
    else:
        boxes = getattr(det_result, "dt_polys", [])

    polys = []
    for box in boxes:
        box = np.asarray(box, dtype=np.float32)
        if box.shape == (4, 2):
            polys.append(box)
    return polys


def _predict_vietocr_batch(crops):
    if not crops:
        return []

    if hasattr(predictor, "predict_batch"):
        return [text.strip() for text in predictor.predict_batch(crops)]

    return [predictor.predict(crop).strip() for crop in crops]


def parse_qwen_ocr(text):
    text = text.strip()

    try:
        obj = json.loads(text)
        if isinstance(obj, str):
            obj = json.loads(obj)
        if isinstance(obj, list):
            return [str(item).strip() for item in obj if str(item).strip()]
    except Exception:
        pass

    return []


if not CFG.use_ocr:

    def get_ocr_batch(image_paths):
        return [[] for _ in image_paths]


    def get_ocr(image_path):
        return []

elif OCR_MODEL == "vietocr":

    def get_ocr_batch(image_paths, det_limit=960):
        image_paths = [str(path) for path in image_paths]
        outputs_by_path = {path: [] for path in image_paths}

        for path_batch in chunked(image_paths, CFG.ocr_det_batch_size):
            images_rgb = {}
            readable_paths = []
            for path in path_batch:
                img = cv2.imread(path)
                if img is None:
                    continue
                images_rgb[path] = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                readable_paths.append(path)

            if not readable_paths:
                continue

            det_results = _predict_text_detector(readable_paths)

            crops = []
            crop_paths = []
            for path, det_result in zip(readable_paths, det_results):
                polys = _extract_polys(det_result)
                if not polys:
                    continue

                line_boxes = merge_boxes_by_line(
                    polys,
                    y_thresh=35,
                    x_gap_thresh=180,
                )

                for box in line_boxes:
                    crop = crop_xyxy(images_rgb[path], box, pad=12)
                    if crop is None or crop.shape[0] < 5 or crop.shape[1] < 5:
                        continue
                    crops.append(Image.fromarray(crop))
                    crop_paths.append(path)

            for crop_batch, crop_path_batch in zip(
                chunked(crops, CFG.ocr_recog_batch_size),
                chunked(crop_paths, CFG.ocr_recog_batch_size),
            ):
                texts = _predict_vietocr_batch(crop_batch)
                for path, text in zip(crop_path_batch, texts):
                    if text:
                        outputs_by_path[path].append(text)

        return [outputs_by_path[path] for path in image_paths]


    def get_ocr(img_path, det_limit=960):
        return get_ocr_batch([img_path], det_limit=det_limit)[0]

elif OCR_MODEL == "qwen":

    def get_ocr_batch(image_paths):
        results = []
        for path_batch in chunked(image_paths, CFG.qwen_ocr_batch_size):
            raw_outputs = _qwen_generate_batch(
                path_batch,
                OCR_PROMPT,
                max_new_tokens=CFG.ocr_max_new_tokens,
            )
            results.extend(parse_qwen_ocr(text) for text in raw_outputs)
        return results


    def get_ocr(image_path):
        return get_ocr_batch([image_path])[0]

### Image captioning

In [ ]:
if not CFG.use_caption:

    def get_caption_batch(image_paths):
        return ["" for _ in image_paths]


    def get_caption(image_path):
        return ""

elif CAPTIONING_MODEL == "qwen":

    def get_caption_batch(image_paths):
        captions = []
        for path_batch in chunked(image_paths, CFG.caption_batch_size):
            captions.extend(
                _qwen_generate_batch(
                    path_batch,
                    CAPTIONING_PROMPT,
                    max_new_tokens=CFG.caption_max_new_tokens,
                )
            )
        return [caption.strip() for caption in captions]


    def get_caption(image_path):
        return get_caption_batch([image_path])[0]

elif CAPTIONING_MODEL == "blip":

    def get_caption_batch(image_paths):
        captions = []
        for path_batch in chunked(image_paths, CFG.caption_batch_size):
            images = [Image.open(path).convert("RGB") for path in path_batch]
            inputs = processor(
                images=images,
                text=[CAPTIONING_PROMPT] * len(images),
                return_tensors="pt",
                padding=True,
            ).to(device, torch.float16)

            with torch.inference_mode():
                generated_ids = blip.generate(
                    **inputs,
                    max_new_tokens=CFG.caption_max_new_tokens,
                    min_new_tokens=CFG.blip_min_new_tokens,
                    num_beams=CFG.blip_num_beams,
                    repetition_penalty=CFG.blip_repetition_penalty,
                    length_penalty=CFG.blip_length_penalty,
                    use_cache=True,
                )

            captions.extend(processor.batch_decode(generated_ids, skip_special_tokens=True))
        return [caption.strip() for caption in captions]


    def get_caption(image_path):
        return get_caption_batch([image_path])[0]

### Object Detection

In [ ]:
def _format_yolo_result(image_path, result):
    record = {
        "image_path": str(image_path),
        "image_name": Path(image_path).name,
        "objects": [],
    }

    for box in result.boxes:
        record["objects"].append({
            "label": yolo.names[int(box.cls.item())],
            "confidence": round(float(box.conf.item()), 4),
        })

    return record


def get_obj_batch(image_paths):
    image_paths = [str(path) for path in image_paths]
    if not CFG.use_objects:
        return [
            {"image_path": path, "image_name": Path(path).name, "objects": []}
            for path in image_paths
        ]

    records = []
    for path_batch in chunked(image_paths, CFG.yolo_batch_size):
        results = yolo(
            path_batch,
            batch=len(path_batch),
            imgsz=CFG.yolo_imgsz,
            # quantize=(device == "cuda"),
            quantize="fp16" if device == "cuda" else "fp32",
            verbose=False,
        )
        records.extend(
            _format_yolo_result(path, result)
            for path, result in zip(path_batch, results)
        )

    return records


def get_obj(image_path: str):
    return get_obj_batch([image_path])[0]

In [ ]:
from collections import Counter
from pathlib import Path
import time


def _combine_feature_record(image_path, caption, texts, obj_result):
    detections = obj_result.get("objects", [])
    labels = [obj["label"] for obj in detections]
    object_counts = dict(Counter(labels))

    return {
        "image_path": str(image_path),
        "image_name": Path(image_path).name,
        "caption": caption,
        "texts": texts,
        "objects": sorted(object_counts.keys()),
        "object_counts": object_counts,
        "detections": detections,
    }


def parse_qwen_combined(text):
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

    try:
        obj = json.loads(text)
        if isinstance(obj, str):
            obj = json.loads(obj)
        caption = str(obj.get("caption", "")).strip() if isinstance(obj, dict) else ""
        texts = obj.get("texts", []) if isinstance(obj, dict) else []
        if not isinstance(texts, list):
            texts = []
        texts = [str(item).strip() for item in texts if str(item).strip()]
        return caption, texts
    except Exception:
        return "", []


def get_qwen_caption_ocr_batch(image_paths):
    captions = []
    texts_by_image = []
    for path_batch in chunked(image_paths, CFG.combined_qwen_batch_size):
        raw_outputs = _qwen_generate_batch(
            path_batch,
            QWEN_COMBINED_PROMPT,
            max_new_tokens=CFG.combined_max_new_tokens,
        )
        parsed = [parse_qwen_combined(text) for text in raw_outputs]
        captions.extend(caption for caption, _ in parsed)
        texts_by_image.extend(texts for _, texts in parsed)
    return captions, texts_by_image


def _should_use_combined_qwen():
    return (
        CFG.combine_qwen_caption_ocr
        and CFG.use_caption
        and CFG.use_ocr
        and CAPTIONING_MODEL == "qwen"
        and OCR_MODEL == "qwen"
    )


def get_all_batch(image_paths, verbose=True):
    image_paths = [str(path) for path in image_paths]
    total_start = time.perf_counter()

    if _should_use_combined_qwen():
        t0 = time.perf_counter()
        captions, texts_by_image = get_qwen_caption_ocr_batch(image_paths)
        qwen_time = time.perf_counter() - t0
        caption_time = qwen_time
        ocr_time = 0.0
    else:
        t0 = time.perf_counter()
        captions = get_caption_batch(image_paths)
        caption_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        texts_by_image = get_ocr_batch(image_paths)
        ocr_time = time.perf_counter() - t0
        qwen_time = None

    t0 = time.perf_counter()
    obj_results = get_obj_batch(image_paths)
    obj_time = time.perf_counter() - t0

    if verbose:
        total_time = time.perf_counter() - total_start
        if qwen_time is None:
            timing = f"caption={caption_time:.3f}s, ocr={ocr_time:.3f}s"
        else:
            timing = f"caption+ocr={qwen_time:.3f}s"
        print(
            f"[Batch {len(image_paths)}] "
            f"{timing}, "
            f"objects={obj_time:.3f}s, "
            f"total={total_time:.3f}s"
        )

    return [
        _combine_feature_record(path, caption, texts, obj_result)
        for path, caption, texts, obj_result in zip(
            image_paths,
            captions,
            texts_by_image,
            obj_results,
        )
    ]


def get_all(image_path):
    return get_all_batch([image_path], verbose=True)[0]

### Test

In [ ]:
Image.open(test_image_path2)

In [ ]:
%%time
get_all(test_image_path2)

In [ ]:
# from pathlib import Path
# import json
# from tqdm.auto import tqdm

# FRAME_ROOT = CFG.frame_root
# OUTPUT_PATH = CFG.output_path
# video_ids = CFG.video_ids

# image_paths = []
# for vid in video_ids:
#     image_paths.extend(sorted((FRAME_ROOT / vid).glob("*.jpg")))

# processed_paths = set()
# if CFG.skip_existing and not CFG.overwrite_output and OUTPUT_PATH.exists():
#     with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
#         for line in f:
#             try:
#                 record = json.loads(line)
#                 if "error" not in record and record.get("image_path"):
#                     processed_paths.add(record["image_path"])
#             except Exception:
#                 pass

# if processed_paths:
#     image_paths = [path for path in image_paths if str(path) not in processed_paths]

# print("Remaining images:", len(image_paths))
# print("Skipped existing:", len(processed_paths))
# OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
# write_mode = "w" if CFG.overwrite_output else "a"
# num_batches = (len(image_paths) + CFG.pipeline_batch_size - 1) // CFG.pipeline_batch_size

# with open(OUTPUT_PATH, write_mode, encoding="utf-8") as f:
#     for batch_idx, path_batch in enumerate(
#         tqdm(chunked(image_paths, CFG.pipeline_batch_size), total=num_batches, desc="Feature batches"),
#         start=1,
#     ):
#         try:
#             records = get_all_batch(path_batch, verbose=True)
#             for record, image_path in zip(records, path_batch):
#                 record["video_id"] = Path(image_path).parent.name
#                 f.write(json.dumps(record, ensure_ascii=False) + "\n")

#         except RuntimeError as e:
#             if "out of memory" in str(e).lower() and torch.cuda.is_available():
#                 torch.cuda.empty_cache()
#             for image_path in path_batch:
#                 error_record = {
#                     "image_path": str(image_path),
#                     "video_id": Path(image_path).parent.name,
#                     "error": str(e),
#                 }
#                 f.write(json.dumps(error_record, ensure_ascii=False) + "\n")

#         except Exception as e:
#             for image_path in path_batch:
#                 error_record = {
#                     "image_path": str(image_path),
#                     "video_id": Path(image_path).parent.name,
#                     "error": str(e),
#                 }
#                 f.write(json.dumps(error_record, ensure_ascii=False) + "\n")

#         if batch_idx % CFG.save_every_n_batches == 0:
#             f.flush()

# print("Saved to:", OUTPUT_PATH)

In [ ]:
from pathlib import Path
import json
from tqdm.auto import tqdm

FRAME_ROOT = CFG.frame_root
OUTPUT_ROOT = CFG.output_root
video_ids = CFG.video_ids

for vid in video_ids:
    video_dir = FRAME_ROOT / vid
    output_dir = OUTPUT_ROOT / vid
    output_path = output_dir / "annotations.jsonl"

    image_paths = sorted(video_dir.glob("*.jpg"))

    processed_paths = set()
    if CFG.skip_existing and not CFG.overwrite_output and output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line)
                    if "error" not in record and record.get("image_path"):
                        processed_paths.add(record["image_path"])
                except Exception:
                    pass

    if processed_paths:
        image_paths = [path for path in image_paths if str(path) not in processed_paths]

    print(f"\nVideo: {vid}")
    print("Remaining images:", len(image_paths))
    print("Skipped existing:", len(processed_paths))

    output_dir.mkdir(parents=True, exist_ok=True)
    write_mode = "w" if CFG.overwrite_output else "a"
    num_batches = (len(image_paths) + CFG.pipeline_batch_size - 1) // CFG.pipeline_batch_size

    with open(output_path, write_mode, encoding="utf-8") as f:
        for batch_idx, path_batch in enumerate(
            tqdm(
                chunked(image_paths, CFG.pipeline_batch_size),
                total=num_batches,
                desc=f"Feature batches {vid}",
            ),
            start=1,
        ):
            try:
                records = get_all_batch(path_batch, verbose=True)

                for record, image_path in zip(records, path_batch):
                    record["video_id"] = Path(image_path).parent.name
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")

            except RuntimeError as e:
                if "out of memory" in str(e).lower() and torch.cuda.is_available():
                    torch.cuda.empty_cache()

                for image_path in path_batch:
                    error_record = {
                        "image_path": str(image_path),
                        "video_id": Path(image_path).parent.name,
                        "error": str(e),
                    }
                    f.write(json.dumps(error_record, ensure_ascii=False) + "\n")

            except Exception as e:
                for image_path in path_batch:
                    error_record = {
                        "image_path": str(image_path),
                        "video_id": Path(image_path).parent.name,
                        "error": str(e),
                    }
                    f.write(json.dumps(error_record, ensure_ascii=False) + "\n")

            if batch_idx % CFG.save_every_n_batches == 0:
                f.flush()

    print("Saved to:", output_path)

In [ ]:
!cd /kaggle/working && zip -r working.zip .